# Normativa docente de Catamarca: preparación del corpus

Este notebook arma el corpus para un sistema de preguntas y respuestas sobre la normativa docente
de Catamarca. La idea es que un docente o alguien de la administración escolar pueda preguntar,
por ejemplo, cómo se cubre una suplencia o qué licencia le corresponde, y reciba la norma y el
artículo que lo regulan, siempre que esa norma siga vigente.

Se trabaja con 13 normas del Digesto de Catamarca, de 1976 a 2025. El notebook las descarga,
extrae y limpia el texto, oculta los DNI, hace un análisis exploratorio, separa las normas que
vienen juntas en la página del Estatuto y busca entidades con expresiones regulares. Partir las
normas en fragmentos para la búsqueda (*chunking*), calcular embeddings y generar respuestas queda
para las etapas siguientes.

**Fuente:** Digesto de Catamarca (`https://digesto.catamarca.gob.ar`), a partir de una copia
guardada el 10/11/2026 (sección 2).

**Cómo ejecutarlo:** en Google Colab, `Runtime → Run all`. Los resultados quedan en
`/content/normativa_docente/` (carpetas `artifacts/` y `figuras/`).

## 1. Setup

Instalación de librerías y carpetas de trabajo.

In [ ]:
# pillow: pdfplumber 0.11.10 exige >=12.2 y el Gradio preinstalado en Colab exige <13
!pip install -q pdfplumber pypdf beautifulsoup4 requests tiktoken matplotlib pandas openpyxl pymupdf nltk "pillow>=12.2,<13" scikit-learn
print("Dependencias instaladas")

In [ ]:
from pathlib import Path
import re, json, hashlib, time, unicodedata, io, csv
from datetime import datetime, timezone

import requests
from bs4 import BeautifulSoup
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import pdfplumber
from pypdf import PdfReader
import tiktoken
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from collections import Counter
import nltk
from nltk.corpus import stopwords
from nltk.util import ngrams

DIGESTO = "https://digesto.catamarca.gob.ar/"
# Instantánea del Digesto publicada en el repositorio del proyecto (archivos y manifiesto con SHA-256)
INSTANTANEA_URL = "https://raw.githubusercontent.com/Alberto919/diiaa-trabajo-final/master/corte_a/fuente_digesto/"
EXPECTED_DOCS = 13
ROOT = Path("/content/normativa_docente")
RAW = ROOT / "raw"
TEXT = ROOT / "text"
CLEAN = ROOT / "clean"
ART = ROOT / "artifacts"
FIGS = ROOT / "figuras"
INSTANTANEA_LOCAL = ROOT / "fuente_digesto"  # si existe (copia local del repositorio), se usa en lugar de la URL
for d in [RAW, TEXT, CLEAN, ART, FIGS]:
    d.mkdir(parents=True, exist_ok=True)

HEADERS = {"User-Agent": "TF-IA-Catamarca/1.0 (academic research)"}
nltk.download("stopwords", quiet=True)
stop_es = set(stopwords.words("spanish"))  # palabras vacías del español (NLTK)
NEGACIONES = {"no", "ni", "sin", "nunca", "tampoco"}  # cambian el sentido de una norma: se preservan siempre
FUENTE = "Fuente: Digesto de Catamarca (instantánea del 2026-11-10)"
print("Raíz:", ROOT)

from importlib.metadata import version
VERSIONS = {pkg: version(pkg) for pkg in [
    "requests", "beautifulsoup4", "pandas", "numpy", "matplotlib", "pdfplumber",
    "pypdf", "pymupdf", "tiktoken", "scikit-learn", "nltk",
]}
print("Versiones:", VERSIONS)

## 2. Adquisición

La tabla `NORMAS` tiene las 13 normas con su ubicación en el Digesto, el tipo, el estado que se usa
en el corpus y su relación con el Estatuto del Docente (Decreto Ley 3122). El estado que muestra la
ficha del Digesto se guarda en otra columna (`estado_digesto`), porque no siempre está al día: la
ficha del Decreto Acuerdo 2010/2013 dice «Vigente», pero el artículo 4 del Decreto Acuerdo 533/2021
lo deroga.

El Digesto tiene una protección contra robots (Cloudflare) que bloquea las conexiones desde Colab
con un error 403, y tampoco deja guardar sus páginas en Wayback Machine. Por eso las normas se
descargaron desde una computadora personal el 10/11/2026 y se guardaron en la carpeta
`fuente_digesto` del repositorio, con un manifiesto que registra la URL de origen, la fecha y el
hash SHA-256 de cada archivo. El notebook baja esos archivos y se detiene si algún hash no
coincide.

Si se ejecuta en una máquina donde el Digesto sí responde, además compara cada norma con la versión
publicada ese día y avisa si alguna cambió.

In [ ]:
# doc_id, norma_id, tipo, ruta del texto en el Digesto, número, patrón del título, estado decidido,
# relación con el Estatuto (Decreto Ley 3122) y artículo del Estatuto que afecta
NORMAS = [
    ("estatuto_dl3122", "Decreto Ley 3122", "Decreto Ley", "ley/ley_detail/713", "3122", r"Decreto Ley N[º°] ?3122", "Vigente", "base", ""),
    ("ley5301", "Ley 5301", "Ley", "ley/ley_detail/146", "5301", r"Ley N[º°] ?5301", "Vigente", "modifica", "3"),
    ("dl3267", "Decreto Ley 3267", "Decreto Ley", "ley/ley_detail/2541", "3267", r"LEY N[º°] ?3267", "Vigente", "modifica", "7"),
    ("ley5852", "Ley 5852", "Ley", "digesto/crearpdf/ley/4514", "5852", r"Ley N[º°] ?5852", "Vigente", "modifica", "74"),
    ("ley4789", "Ley 4789", "Ley", "digesto/crearpdf/ley/1819", "4789", r"Ley N[º°] ?4789", "Vigente", "modifica", "28"),
    ("ley5103", "Ley 5103", "Ley", "digesto/crearpdf/ley/355", "5103", r"Ley N[º°] ?5103", "Vigente", "independiente", ""),
    ("ley5381", "Ley 5381", "Ley", "digesto/crearpdf/ley/196", "5381", r"Ley N[º°] ?5381", "Vigente", "independiente", ""),
    ("da1385_92", "Decreto Acuerdo 1385/92", "Decreto Acuerdo", "digesto/crearpdf/decreto/5205", "1385", r"Decreto Acuerdo N[º°] ?1385", "Vigente", "reglamenta", ""),
    ("da1092_15", "Decreto Acuerdo 1092/2015", "Decreto Acuerdo", "decreto/decreto_detail/3678", "1092", r"Decreto Acuerdo N[º°] ?1092", "Vigente", "reglamenta", ""),
    ("da2010_13", "Decreto Acuerdo 2010/2013", "Decreto Acuerdo", "digesto/crearpdf/decreto/4404", "2010", r"Decreto Acuerdo N[º°] ?2010", "Derogado", "reglamenta", ""),
    ("da533_21", "Decreto Acuerdo 533/2021", "Decreto Acuerdo", "digesto/crearpdf/decreto/3874", "533", r"Decreto Acuerdo N[º°] ?533", "Vigente", "reglamenta", ""),
    ("da636_21", "Decreto Acuerdo 636/2021", "Decreto Acuerdo", "digesto/crearpdf/decreto/3867", "636", r"Decreto Acuerdo N[º°] ?636", "Vigente", "reglamenta", ""),
    ("d928_25", "Decreto 928/2025", "Decreto", "decreto/decreto_detail/5202", "928", r"N[º°] ?928", "Vigente", "reglamenta", ""),
]
df_normas = pd.DataFrame(NORMAS, columns=["doc_id", "norma_id", "tipo", "ruta", "numero", "patron_titulo",
                                          "estado", "relacion_3122", "articulo_3122"])
assert len(df_normas) == EXPECTED_DOCS and df_normas["doc_id"].is_unique
df_normas[["doc_id", "norma_id", "tipo", "estado", "relacion_3122", "articulo_3122"]]

In [ ]:
def sha256_bytes(b: bytes) -> str:
    return hashlib.sha256(b).hexdigest()


def leer_instantanea(nombre: str) -> bytes:
    """Archivo de la instantánea: de la copia local si existe; si no, del repositorio publicado."""
    local = INSTANTANEA_LOCAL / nombre
    if local.exists():
        return local.read_bytes()
    for intento in range(3):
        try:
            r = requests.get(INSTANTANEA_URL + nombre, headers=HEADERS, timeout=60)
            r.raise_for_status()
            return r.content
        except requests.RequestException as e:
            ultimo = e
            time.sleep(2 * (intento + 1))
    raise RuntimeError(f"No se pudo descargar {INSTANTANEA_URL + nombre}: {ultimo}. "
                       "¿Está publicado el repositorio del proyecto?")


def texto_norma(b: bytes) -> str:
    """Texto de una norma (PDF o página HTML del Digesto), sin la línea de fecha de impresión."""
    if b[:4] == b"%PDF":
        with pdfplumber.open(io.BytesIO(b)) as pdf:
            t = "\n".join(p.extract_text() or "" for p in pdf.pages)
    else:
        div = BeautifulSoup(b, "html.parser").find(id="desplegable")  # bloque del cuerpo de la norma
        t = div.get_text("\n") if div else ""
    t = re.sub(r"\*+\s*Esta normativa fue impresa desde el Digesto Catamarca[^\n]*", "", t)
    return re.sub(r"\s+", " ", t).strip()


def estado_ficha(b: bytes) -> str | None:
    s = BeautifulSoup(b, "html.parser").find("strong", string=re.compile("Estado:"))
    return re.sub(r"\s+", " ", s.parent.get_text(" ")).replace("Estado:", "").strip() if s else None


# 1) Manifiesto de la instantánea y verificación de cada archivo por SHA-256
man_inst = pd.read_csv(io.BytesIO(leer_instantanea("manifiesto_fuente.csv")), dtype=str)
hash_inst = {}
for r in man_inst.itertuples():
    b = leer_instantanea(r.archivo)
    hash_inst[r.archivo] = sha256_bytes(b) == r.sha256
    if hash_inst[r.archivo]:
        (RAW / r.archivo).write_bytes(b)
malos = [a for a, ok in hash_inst.items() if not ok]
assert not malos, f"Archivos de la instantánea con hash distinto al del manifiesto: {malos}"
FECHA_INSTANTANEA = man_inst["fecha_descarga"].iloc[0]
print(f"Instantánea del {FECHA_INSTANTANEA}: {len(hash_inst)} archivos, todos con el SHA-256 del manifiesto.")

# 2) Control de vigencia de la instantánea: comparar con el Digesto si responde
def digesto_responde() -> bool:
    try:
        r = requests.get(DIGESTO + NORMAS[1][3], headers=HEADERS, timeout=30)
        return r.status_code == 200 and b"Just a moment" not in r.content[:3000]
    except requests.RequestException:
        return False


ACCESO_DIGESTO = digesto_responde()
print("El Digesto responde desde este entorno:" if ACCESO_DIGESTO else
      "El Digesto no responde desde este entorno (verificación anti-bots): no se puede comparar con el texto del día.",
      ACCESO_DIGESTO if ACCESO_DIGESTO else "")

# 3) Manifiesto del corpus
manifest = []
for n in df_normas.itertuples():
    fila_t = man_inst[(man_inst["doc_id"] == n.doc_id) & (man_inst["rol"] == "texto")].iloc[0]
    fila_f = man_inst[(man_inst["doc_id"] == n.doc_id) & (man_inst["rol"] == "ficha")]
    b = (RAW / fila_t["archivo"]).read_bytes()
    ficha = (RAW / fila_f.iloc[0]["archivo"]).read_bytes() if len(fila_f) else b
    texto = texto_norma(b)
    control = "sin acceso al Digesto"
    if ACCESO_DIGESTO:
        try:
            hoy = requests.get(DIGESTO + n.ruta, headers=HEADERS, timeout=60).content
            control = "igual" if texto_norma(hoy) == texto else "CAMBIÓ: revisar y renovar la instantánea"
        except requests.RequestException:
            control = "sin acceso al Digesto"
        time.sleep(1)
    formato = "pdf" if b[:4] == b"%PDF" else "html"
    manifest.append({
        "doc_id": n.doc_id, "norma_id": n.norma_id, "tipo": n.tipo, "titulo": texto[:120],
        "url_origen": DIGESTO + n.ruta, "fuente": "Digesto Catamarca", "fuente_descarga": "instantanea",
        "fecha_instantanea": FECHA_INSTANTANEA, "archivo": str(RAW / fila_t["archivo"]), "formato": formato,
        "bytes": len(b), "sha256": sha256_bytes(b),
        "paginas": len(PdfReader(io.BytesIO(b)).pages) if formato == "pdf" else None,
        "estado_digesto": estado_ficha(ficha), "estado": n.estado,
        "relacion_3122": n.relacion_3122, "articulo_3122": n.articulo_3122,
        "titulo_esperado": bool(re.search(n.patron_titulo, texto[:600], re.I)),
        "control_vigencia": control,
    })
df_manifest = pd.DataFrame(manifest)
df_manifest.to_csv(ART / "manifest.csv", index=False)
df_manifest.to_json(ART / "manifest.json", orient="records", force_ascii=False, indent=2)

assert df_manifest["titulo_esperado"].all(), df_manifest.loc[~df_manifest["titulo_esperado"], "doc_id"].tolist()
difieren = df_manifest[df_manifest["estado"] != df_manifest["estado_digesto"]]
print("Normas cuyo estado decidido difiere de la ficha del Digesto:")
print(difieren[["norma_id", "estado_digesto", "estado"]].to_string(index=False))
print("Control de vigencia:", df_manifest["control_vigencia"].value_counts().to_dict())
df_manifest[["doc_id", "norma_id", "formato", "paginas", "estado_digesto", "estado", "relacion_3122", "control_vigencia"]]

## 3. Extracción de texto

De las páginas HTML se toma solo el bloque con el texto de la norma, sin el menú ni el formulario
de comentarios del sitio. Los PDF se leen página por página con `pdfplumber`, para conservar el
número de página, que después sirve para citar. La celda controla que todas las normas tengan
texto.

In [ ]:
def extract_pdf_pages(path: str | Path) -> list[dict]:
    pages = []
    with pdfplumber.open(str(path)) as pdf:
        for i, page in enumerate(pdf.pages, 1):
            text = page.extract_text() or ""
            pages.append({"page": i, "text": text, "chars": len(text.strip())})
    return pages


def extract_html(path: str | Path) -> list[dict]:
    div = BeautifulSoup(Path(path).read_bytes(), "html.parser").find(id="desplegable")
    text = div.get_text("\n") if div else ""
    return [{"page": None, "text": text, "chars": len(text.strip())}]


INTERFAZ = ["envianos tu comentario", "captcha", "lo estamos haciendo entre todos"]
corpus = []
for row in df_manifest.itertuples():
    pages = extract_pdf_pages(row.archivo) if row.formato == "pdf" else extract_html(row.archivo)
    texto = "\n".join(p["text"] for p in pages)
    payload = {
        "doc_id": row.doc_id, "norma_id": row.norma_id, "titulo": row.titulo, "url": row.url_origen,
        "formato": row.formato, "pages": pages, "chars_total": sum(p["chars"] for p in pages),
        "pages_empty": sum(1 for p in pages if p["chars"] < 20),
        "restos_interfaz": sum(texto.lower().count(x) for x in INTERFAZ),
    }
    (TEXT / f"{row.doc_id}.json").write_text(json.dumps(payload, ensure_ascii=False, indent=2), encoding="utf-8")
    corpus.append(payload)

df_text = pd.DataFrame([
    {"doc_id": p["doc_id"], "formato": p["formato"], "paginas": len(p["pages"]) if p["formato"] == "pdf" else None,
     "chars_total": p["chars_total"], "paginas_vacias": p["pages_empty"], "restos_interfaz": p["restos_interfaz"],
     "sin_texto": p["chars_total"] < 200}
    for p in corpus
])
df_text.to_csv(ART / "text_layer.csv", index=False)
print("Normas sin texto extraíble:", df_text.loc[df_text["sin_texto"], "doc_id"].tolist() or "ninguna")
print("Restos de la interfaz del sitio en el texto:", int(df_text["restos_interfaz"].sum()))
df_text

## 4. Limpieza

Los PDF del Digesto traen un encabezado («Gobierno de Catamarca – Asesoría General de Gobierno») y
un pie con la fecha de impresión que no son parte de la norma, así que se sacan. También se
normalizan los caracteres y se unen las palabras cortadas con guion al final de línea. Se mide
cuánto texto se pierde (no debería pasar del 5 %) y se controla que no desaparezcan negaciones,
números de artículo ni fechas.

Algunas normas nombran personas con su DNI. En esta misma etapa los números de DNI se reemplazan
por X, así ningún archivo que se genera después los contiene.

In [ ]:
HEADER_LINES = [
    r"^gobierno de catamarca\s*$",
    r"^asesor[ií]a general de gobierno\s*$",
    r"^\*+\s*esta normativa fue impresa desde el digesto catamarca.*$",
    r"^ministerio de planificaci[oó]n y modernizaci[oó]n\s*$",
    r"^secretar[ií]a de modernizaci[oó]n del estado.*$",
]

# DNI con o sin puntos, precedido de DNI, D.N.I., M.I. o L.C.
DNI_RE = re.compile(r"\b(?:D\.?\s?N\.?\s?I\.?|M\.?\s?I\.?|L\.?\s?C\.?)\s*(?:N[°º]\.?|Nro\.?)?\s*:?\s*\d{1,2}\.?\d{3}\.?\d{3}\b", re.I)


def ocultar_dni(text: str) -> tuple[str, int]:
    n = len(DNI_RE.findall(text))
    return DNI_RE.sub(lambda m: re.sub(r"\d", "X", m.group(0)), text), n


MAX_LOSS_PCT = 5.0  # la limpieza no debe perder más del 5 % del texto
PRESERVE = {
    "negaciones": r"\b(?:no|sin|nunca|ni)\b",
    "articulos": r"ART[IÍ]CULO\s+\d+",
    "fechas": r"\b\d{1,2}/\d{1,2}/\d{2,4}\b|\b\d{1,2}\s+de\s+[a-záéíóú]+\s+(?:de\s+)?\d\.?\d{3}\b",
}


def nonblank(s: str) -> int:
    return len(re.sub(r"\s+", "", s))


BOILERPLATE = {"chars": 0}  # caracteres no blancos de membrete quitados en la última llamada


def clean_text(text: str) -> str:
    BOILERPLATE["chars"] = 0
    text = unicodedata.normalize("NFC", text or "")
    text = text.replace("\u00ad", "").replace("\u00a0", " ").replace("\u2003", " ").replace("\r", "")
    text = re.sub(r"(?<=\w)-\n(?=\w)", "", text)
    lines = []
    for line in text.splitlines():
        s = line.strip()
        if not s:
            lines.append("")
            continue
        if any(re.match(p, s, re.I) for p in HEADER_LINES):
            BOILERPLATE["chars"] += nonblank(s)
            continue
        lines.append(s)
    text = "\n".join(lines)
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()


corpus = [json.loads(p.read_text(encoding="utf-8")) for p in sorted(TEXT.glob("*.json"))]
clean_rows = []
for payload in corpus:
    doc_id = payload["doc_id"]
    raw_text = "\n".join(p["text"] for p in payload["pages"])
    cleaned, n_dni = ocultar_dni(clean_text(raw_text))
    (CLEAN / f"{doc_id}.txt").write_text(cleaned, encoding="utf-8")
    row = {
        "doc_id": doc_id,
        "chars_raw": len(raw_text),
        "dni_ocultos": n_dni,
        "chars_clean": len(cleaned),
        # Sobre caracteres no blancos (normalizar espacios no es pérdida):
        # - pct_boilerplate: membrete quitado a propósito (se informa, no es pérdida de información)
        # - pct_lost: pérdida residual, la que no se explica por el membrete (umbral ≤ 5 %)
        "pct_boilerplate": round(100 * BOILERPLATE["chars"] / max(1, nonblank(raw_text)), 2),
        "pct_lost": round(100 * (1 - (nonblank(cleaned) + BOILERPLATE["chars"]) / max(1, nonblank(raw_text))), 2),
    }
    # Lo que la limpieza NO debe perder: negaciones, artículos y fechas
    for name, pat in PRESERVE.items():
        row[f"{name}_antes"] = len(re.findall(pat, raw_text, re.I))
        row[f"{name}_despues"] = len(re.findall(pat, cleaned, re.I))
    clean_rows.append(row)

df_clean = pd.DataFrame(clean_rows)
df_clean["pierde_mas_5pct"] = df_clean["pct_lost"] > MAX_LOSS_PCT
df_clean["preserva_todo"] = [
    all(r[f"{n}_antes"] == r[f"{n}_despues"] for n in PRESERVE) for _, r in df_clean.iterrows()
]
print(f"Membrete quitado: {df_clean['pct_boilerplate'].min():.1f}–{df_clean['pct_boilerplate'].max():.1f} % por documento (intencional).")
print(f"Pérdida residual máxima: {df_clean['pct_lost'].max():.2f} % (umbral {MAX_LOSS_PCT} %)")
print("Documentos sobre el umbral:", df_clean.loc[df_clean["pierde_mas_5pct"], "doc_id"].tolist() or "ninguno")
print(f"DNI ocultos: {int(df_clean['dni_ocultos'].sum())} en "
      f"{df_clean.loc[df_clean['dni_ocultos'] > 0, 'doc_id'].tolist() or 'ninguna norma'}")
print("Documentos que pierden negaciones, artículos o fechas:",
      df_clean.loc[~df_clean["preserva_todo"], "doc_id"].tolist() or "ninguno")
df_clean.to_csv(ART / "cleaning_stats.csv", index=False)
df_clean

In [ ]:
example = next(p for p in corpus if "estatuto" in p["doc_id"].lower())
raw = "\n".join(p["text"] for p in example["pages"])
clean = (CLEAN / f"{example['doc_id']}.txt").read_text(encoding="utf-8")
print("ANTES (primeros 500 caracteres):\n")
print(raw[:500])
print("\nDESPUÉS (primeros 500 caracteres):\n")
print(clean[:500])

## 5. Análisis exploratorio (EDA)

El texto de una norma no se grafica tal cual: primero se convierte en variables que sí se pueden
contar o comparar. Estas son las que usa el análisis, con su tipo:

| Variable | Tipo | Ejemplo |
|---|---|---|
| Tokens por norma y por artículo | numérica discreta | 108 tokens (mediana de los artículos) |
| Tokens por carácter, similitud coseno, TTR y MATTR | numérica continua | 0,30 tokens por carácter |
| Tipo de norma | categórica nominal | Ley, Decreto, Decreto Acuerdo, Decreto Ley |
| Relación con el Estatuto | categórica nominal | base, modifica, reglamenta, independiente |
| Tipo de entidad | categórica nominal | NORMA, ARTICULO, FECHA, CARGO… |
| Estado | binaria | Vigente o Derogado |
| Formato | binaria | página HTML o PDF |
| Tiene nota de modificación | binaria | sí o no |
| Texto de la norma | texto libre | «ARTÍCULO 1º.- Apruébase el Sistema de Asamblea Pública…» |

Cada gráfico tiene antes una celda que dice qué variables usa, qué pregunta responde y por qué es
ese tipo de gráfico; el hallazgo y la decisión se imprimen debajo. No se lematiza ni se sacan las
negaciones. Los tres hallazgos que guían el diseño son el del gráfico 2 (el artículo sirve como
fragmento), el del gráfico 8 (hay normas derogadas que hay que poder excluir) y el del gráfico 9
(la similitud sola no alcanza para detectar duplicados).

In [ ]:
enc = tiktoken.get_encoding("cl100k_base")


def count_articles(text: str) -> int:
    return len(re.findall(r"ART[IÍ]CULO\s+\d+", text, re.I))


eda_rows = []
for payload in corpus:
    doc_id = payload["doc_id"]
    text = (CLEAN / f"{doc_id}.txt").read_text(encoding="utf-8")
    eda_rows.append({
        "doc_id": doc_id,
        "norma_id": payload["norma_id"],
        "formato": payload["formato"],
        "chars": len(text),
        "tokens": len(enc.encode(text)),
        "articles": count_articles(text),
    })

df_eda = pd.DataFrame(eda_rows).merge(df_manifest[["doc_id", "estado"]], on="doc_id").sort_values("tokens", ascending=False)
df_eda["pct_corpus"] = (100 * df_eda["tokens"] / df_eda["tokens"].sum()).round(1)
df_eda.to_csv(ART / "eda_stats.csv", index=False)
df_eda

**Gráfico 1 — Tokens por norma**

- **Variables:** tokens por norma (numérica discreta) y norma (categórica nominal).
- **Pregunta:** ¿cada norma entra entera en los 512 tokens que lee el modelo de embeddings?
- **Gráfico:** barras horizontales ordenadas, porque se compara un valor numérico entre categorías; escala logarítmica porque los valores van de cientos a decenas de miles.
- **Hallazgo y decisión:** se calculan con los datos y se imprimen debajo del gráfico.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
ax.barh(df_eda["doc_id"], df_eda["tokens"], color="#0f766e")
ax.set_xscale("log")
ax.axvline(512, color="#b91c1c", linestyle="--", linewidth=1.2)
ax.set_xlabel("Tokens (cl100k_base, escala log)")
for y, v in zip(df_eda["doc_id"], df_eda["tokens"]):
    ax.text(v * 1.08, y, f"{v:,}".replace(",", "."), va="center", fontsize=7)
ax.set_title(f"{int((df_eda['tokens'] > 512).sum())} de {len(df_eda)} normas superan los 512 tokens: no entran enteras en el modelo")
ax.spines[["top", "right"]].set_visible(False)
fig.text(0.99, 0.01, FUENTE, ha="right", fontsize=7)
ax.invert_yaxis()
plt.tight_layout()
plt.savefig(FIGS / "eda_tokens.png", dpi=200, bbox_inches="tight")
plt.show()

over_512 = int((df_eda["tokens"] > 512).sum())
print(f"Hallazgo: {over_512} de {len(df_eda)} documentos superan 512 tokens de referencia.")
print("Decisión: el chunking es obligatorio; el tamaño se medirá con el tokenizador del modelo de embeddings.")

In [ ]:
# Caracteres vs. tokens: la correlación es trivial (r ≈ 1), por eso no se grafica.
# Lo útil es la razón tokens/carácter para estimar presupuestos de chunking.
valid = df_eda[df_eda["chars"] > 0]
ratio = (valid["tokens"] / valid["chars"]).round(3)
print(ratio.describe().round(3).to_string())
pal_total = sum(len((CLEAN / f"{d}.txt").read_text(encoding="utf-8").split()) for d in df_eda["doc_id"])
print(f"Tokens por palabra en todo el corpus: {df_eda['tokens'].sum() / pal_total:.2f}")
print(f"Hallazgo: la razón tokens/carácter es {ratio.median():.3f} (mediana) y varía poco entre documentos.")
print("Decisión: estimar tokens desde caracteres solo para presupuestos; el chunk final se mide con el "
      "tokenizador del modelo de embeddings. Por eso este cruce no se grafica.")

**Gráfico 2 — Tokens por artículo**

- **Variables:** tokens por artículo (numérica discreta).
- **Pregunta:** ¿el artículo sirve como fragmento, o es demasiado largo para el modelo?
- **Gráfico:** histograma, porque es una sola variable numérica; con líneas en la media, la mediana y los límites de 200 y 512 tokens. La media queda a la derecha de la mediana cuando hay pocos artículos muy largos.
- **Hallazgo y decisión:** se calculan con los datos y se imprimen debajo del gráfico.

In [ ]:
# Unidad candidata de chunking: el artículo. Se mide su longitud en tokens.
ART_SPLIT = re.compile(r"(?m)^\*?\s*ART[IÍ]CULO\s+\d+")

art_rows = []
for payload in corpus:
    doc_id = payload["doc_id"]
    text = (CLEAN / f"{doc_id}.txt").read_text(encoding="utf-8")
    starts = [m.start() for m in ART_SPLIT.finditer(text)]
    for k, s in enumerate(starts):
        e = starts[k + 1] if k + 1 < len(starts) else len(text)
        body = text[s:e]
        art_rows.append({
            "doc_id": doc_id,
            "inicio": body[:90].replace("\n", " "),
            "tokens": len(enc.encode(body)),
            "modificado": body.lstrip().startswith("*") or bool(re.search(r"Modificad[oa]\s+por", body, re.I)),
        })

df_art_tok = pd.DataFrame(art_rows)
df_art_tok.to_csv(ART / "articles_tokens.csv", index=False)

med = df_art_tok["tokens"].median()
p90 = df_art_tok["tokens"].quantile(0.9)
pct_512 = 100 * (df_art_tok["tokens"] > 512).mean()

fig, ax = plt.subplots(figsize=(9, 4.5))
bins = np.logspace(np.log10(max(1, df_art_tok["tokens"].min())), np.log10(df_art_tok["tokens"].max()), 40)
ax.hist(df_art_tok["tokens"], bins=bins, color="#0f766e", alpha=0.8)
ax.set_xscale("log")
for x in (200, 512):
    ax.axvline(x, color="#b91c1c", linestyle="--", linewidth=1)
    ax.text(x, ax.get_ylim()[1] * 0.92, f" {x}", color="#b91c1c")
ax.axvline(med, color="#1d4ed8", linewidth=1.5, label=f"mediana: {med:.0f} tokens")
ax.axvline(df_art_tok["tokens"].mean(), color="#f59e0b", linewidth=1.5, label=f"media: {df_art_tok['tokens'].mean():.0f} tokens")
ax.legend(loc="upper right", fontsize=8)
ax.spines[["top", "right"]].set_visible(False)
ax.set_xlabel("Tokens por segmento de artículo (cl100k_base, escala log)")
ax.set_ylabel("Artículos")
ax.set_title(f"{len(df_art_tok)} artículos: mediana {med:.0f} tokens; {pct_512:.1f} % supera 512")
ax.text(1, -0.18, FUENTE,
        transform=ax.transAxes, ha="right", fontsize=7)
plt.tight_layout()
plt.savefig(FIGS / "eda_tokens_por_articulo.png", dpi=200, bbox_inches="tight")
plt.show()

long_segs = df_art_tok[df_art_tok["tokens"] > 1500].sort_values("tokens", ascending=False)
print("Segmentos de más de 1500 tokens (posibles fallas de segmentación o anexos):")
print(long_segs[["doc_id", "tokens", "inicio"]].head(8).to_string(index=False))

# Valores atípicos de longitud: umbrales de 100 y 800 tokens
UMBRAL_CORTO, UMBRAL_LARGO = 100, 800
df_art_tok["categoria"] = np.where(df_art_tok["tokens"] < UMBRAL_CORTO, "Demasiado corto",
                          np.where(df_art_tok["tokens"] > UMBRAL_LARGO, "Requiere sub-chunking", "Óptimo"))
print(df_art_tok["categoria"].value_counts().to_string())

top = df_eda.iloc[0]
print(f"Hallazgo: {len(df_art_tok)} artículos; mediana {med:.0f} tokens, p90 {p90:.0f}; {pct_512:.1f} % supera 512.")
print(f"Hallazgo: {top['doc_id']} concentra el {top['pct_corpus']} % de los tokens del corpus.")
print(f"Hallazgo: {int(df_art_tok['modificado'].sum())} artículos tienen marca de modificación.")
print(f"Hallazgo: {len(long_segs)} segmentos superan 1500 tokens; el mayor tiene {int(df_art_tok['tokens'].max())}: "
      "hay tramos (anexos, decretos agregados) que el patrón ARTICULO no separa.")
print("Decisión: chunk = artículo; subdividir solo los que superan el límite del modelo de embeddings; "
      "conservar la marca de modificación como metadato; estratificar las preguntas de evaluación por documento.")

**Gráfico 3 — Largo de los artículos según el tipo de norma**

- **Variables:** tokens por artículo (numérica discreta) y tipo de norma (categórica nominal: Ley, Decreto, Decreto Acuerdo, Decreto Ley).
- **Pregunta:** ¿cambia el largo de los artículos según el tipo de norma? Si cambia mucho, convendría fragmentar distinto cada tipo.
- **Gráfico:** diagrama de caja con los puntos encima, porque se cruza una variable numérica con una categórica. La caja muestra el 50 % central y la línea del medio, la mediana.
- **Hallazgo y decisión:** se calculan con los datos y se imprimen debajo del gráfico.

In [ ]:
# Largo de los artículos según el tipo de norma (la página del Estatuto cuenta como Decreto Ley)
df_art_tok = df_art_tok.drop(columns=["tipo"], errors="ignore").merge(df_manifest[["doc_id", "tipo"]], on="doc_id", how="left")
med_tipo = df_art_tok.groupby("tipo")["tokens"].median().sort_values(ascending=False)
datos = [df_art_tok.loc[df_art_tok["tipo"] == t, "tokens"] for t in med_tipo.index]

fig, ax = plt.subplots(figsize=(9, 4.5))
ax.boxplot(datos, showfliers=False)
rng = np.random.default_rng(0)
for k, d in enumerate(datos, 1):
    ax.scatter(rng.normal(k, 0.07, len(d)), d, s=6, alpha=0.3, color="#0f766e")
ax.set_xticks(range(1, len(datos) + 1), [f"{t}\n({len(d)} artículos)" for t, d in zip(med_tipo.index, datos)])
ax.set_yscale("log")
ax.axhline(512, color="#b91c1c", linestyle="--", linewidth=1)
ax.text(len(datos) + 0.45, 512, "512", color="#b91c1c", va="center", fontsize=8)
ax.set_ylabel("Tokens por artículo (cl100k_base, escala log)")
ax.set_title(f"La mediana va de {med_tipo.min():.0f} a {med_tipo.max():.0f} tokens según el tipo: todas quedan lejos de 512")
ax.spines[["top", "right"]].set_visible(False)
fig.text(0.99, 0.01, FUENTE, ha="right", fontsize=7)
plt.tight_layout()
plt.savefig(FIGS / "eda_tokens_por_tipo.png", dpi=200, bbox_inches="tight")
plt.show()

print("Mediana de tokens por artículo según el tipo de norma:", med_tipo.round(0).to_dict())
print(f"Hallazgo: la mediana va de {med_tipo.min():.0f} a {med_tipo.max():.0f} tokens; en todos los tipos, "
      f"la mayoría de los artículos queda por debajo de 512.")
print("Decisión: un mismo criterio de fragmentación para todos los tipos de norma (un artículo por fragmento), "
      "y dividir solo los artículos que superan el límite.")

**Una fila real con todos los tipos de variable.** El texto de una norma no se grafica directamente:
se convierte en variables de los otros tipos (cuántos tokens tiene, de qué tipo de norma es, si
está vigente, si fue modificado). Así se ve un artículo del Decreto Acuerdo 636/2021:

In [ ]:
art = df_art_tok.merge(df_manifest[["doc_id", "estado", "formato", "relacion_3122"]], on="doc_id")
fila = art[art["doc_id"] == "da636_21"].iloc[1]
pd.DataFrame([
    {"tipo de variable": "numérica discreta", "variable": "tokens", "valor": fila["tokens"]},
    {"tipo de variable": "categórica nominal", "variable": "tipo de norma", "valor": fila["tipo"]},
    {"tipo de variable": "categórica nominal", "variable": "relación con el Estatuto", "valor": fila["relacion_3122"]},
    {"tipo de variable": "binaria", "variable": "estado (Vigente / Derogado)", "valor": fila["estado"]},
    {"tipo de variable": "binaria", "variable": "formato (html / pdf)", "valor": fila["formato"]},
    {"tipo de variable": "binaria", "variable": "tiene nota de modificación", "valor": fila["modificado"]},
    {"tipo de variable": "texto libre", "variable": "texto", "valor": fila["inicio"] + "…"},
])

**Gráfico 4 — Palabras vacías**

- **Variables:** porcentaje de palabras vacías por norma (numérica continua) y frecuencia de cada palabra vacía (numérica discreta).
- **Pregunta:** ¿cuánto del texto son palabras vacías, y entre ellas hay palabras que cambian el sentido de una norma?
- **Gráfico:** dos gráficos de barras ordenadas; en rojo, las negaciones («no», «sin», «ni»), que nunca se quitan.
- **Hallazgo y decisión:** se calculan con los datos y se imprimen debajo del gráfico.

In [ ]:
# Densidad de palabras vacías por documento
def palabras(texto: str) -> list[str]:
    return [t for t in re.findall(r"[a-záéíóúüñ]+", texto.lower())]


sw_rows = []
sw_counter = Counter()
for payload in corpus:
    doc_id = payload["doc_id"]
    tok = palabras((CLEAN / f"{doc_id}.txt").read_text(encoding="utf-8"))
    if not tok:
        continue
    sw = [t for t in tok if t in stop_es]
    sw_counter.update(sw)
    sw_rows.append({"doc_id": doc_id, "palabras": len(tok), "pct_vacias": round(100 * len(sw) / len(tok), 1)})
df_sw = pd.DataFrame(sw_rows).sort_values("pct_vacias")

fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
axes[0].barh(df_sw["doc_id"], df_sw["pct_vacias"], color="#0f766e")
axes[0].set_xlabel("Palabras vacías (% de las palabras del documento)")
axes[0].set_title(f"Entre {df_sw['pct_vacias'].min():.0f} % y {df_sw['pct_vacias'].max():.0f} % del texto son palabras vacías")
top_sw = sw_counter.most_common(20)[::-1]
axes[1].barh([w for w, _ in top_sw], [c for _, c in top_sw],
             color=["#b91c1c" if w in NEGACIONES else "#94a3b8" for w, _ in top_sw])
axes[1].set_xlabel("Ocurrencias en el corpus")
axes[1].set_title("Top 20 palabras vacías (rojo = negación, se preserva)")
for a in axes:
    a.spines[["top", "right"]].set_visible(False)
fig.text(0.99, 0.01, FUENTE, ha="right", fontsize=7)
plt.tight_layout()
plt.savefig(FIGS / "eda_palabras_vacias.png", dpi=200, bbox_inches="tight")
plt.show()

neg = {w: sw_counter[w] for w in NEGACIONES if sw_counter[w]}
print(f"Hallazgo: las palabras vacías son el {df_sw['pct_vacias'].mean():.1f} % del texto en promedio; negaciones: {neg}.")
print("Decisión: quitarlas solo en el baseline léxico (TF-IDF/BM25), conservando las negaciones; "
      "no quitarlas antes de los embeddings.")

**Gráfico 5 — Frases que más se repiten**

- **Variables:** pares y tríos de palabras seguidas (n-gramas, derivados del texto libre) y su frecuencia (numérica discreta).
- **Pregunta:** ¿qué frases del dominio no habría que cortar al dividir el texto?
- **Gráfico:** barras horizontales ordenadas de mayor a menor, sin palabras vacías.
- **Hallazgo y decisión:** se calculan con los datos y se imprimen debajo del gráfico.

In [ ]:
# N-gramas de dominio: frases compuestas que no deben separarse
def top_ngrams(texto: str, n: int, k: int = 15):
    tok = [t for t in palabras(texto) if t not in stop_es and len(t) > 1]  # sin letras sueltas («Ley N°» → «n»)
    return Counter(ngrams(tok, n)).most_common(k)


corpus_all_text = "\n".join((CLEAN / f"{p['doc_id']}.txt").read_text(encoding="utf-8") for p in corpus)
bigramas = top_ngrams(corpus_all_text, 2)
trigramas = top_ngrams(corpus_all_text, 3)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, data, titulo in [(axes[0], bigramas, "Top 15 bigramas"), (axes[1], trigramas, "Top 15 trigramas")]:
    etiquetas = [" ".join(g) for g, _ in data][::-1]
    ax.barh(etiquetas, [c for _, c in data][::-1], color="#7c3aed")
    ax.set_xlabel("Ocurrencias (sin palabras vacías)")
    ax.set_title(titulo)
fig.suptitle(f"El dominio repite frases fijas: «{' '.join(bigramas[0][0])}» aparece {bigramas[0][1]} veces")
for a in axes:
    a.spines[["top", "right"]].set_visible(False)
fig.text(0.99, 0.01, FUENTE, ha="right", fontsize=7)
plt.tight_layout()
plt.savefig(FIGS / "eda_ngramas.png", dpi=200, bbox_inches="tight")
plt.show()

NO_SEPARAR = [" ".join(g) for g, _ in bigramas[:10]]
print("Hallazgo: frases de dominio más frecuentes:", NO_SEPARAR)
print("Decisión: no cortar chunks dentro de estas frases y usarlas como vocabulario de control del baseline léxico.")

**Gráfico 6 — Riqueza del vocabulario**

- **Variables:** TTR y MATTR por norma (numéricas continuas, entre 0 y 1) y norma ordenada por cantidad de palabras.
- **Pregunta:** ¿el TTR sirve para comparar normas de distinto largo, o conviene el MATTR?
- **Gráfico:** barras agrupadas: dos medidas numéricas por cada norma, ordenadas de la más corta a la más larga.
- **Hallazgo y decisión:** se calculan con los datos y se imprimen debajo del gráfico.

In [ ]:
# Riqueza léxica: TTR y MATTR con ventana de 50
def ttr(texto: str, ventana: int | None = None) -> float:
    tok = palabras(texto)
    if not tok:
        return 0.0
    if ventana is None:
        return len(set(tok)) / len(tok)
    if len(tok) < ventana:
        return len(set(tok)) / len(tok)
    return float(np.mean([len(set(tok[k:k + ventana])) / ventana for k in range(len(tok) - ventana + 1)]))


ttr_rows = []
for payload in corpus:
    texto = (CLEAN / f"{payload['doc_id']}.txt").read_text(encoding="utf-8")
    if texto.strip():
        ttr_rows.append({"doc_id": payload["doc_id"], "palabras": len(palabras(texto)),
                         "ttr": round(ttr(texto), 3), "mattr_50": round(ttr(texto, 50), 3)})
df_ttr = pd.DataFrame(ttr_rows).sort_values("palabras")

fig, ax = plt.subplots(figsize=(11, 4.5))
x = np.arange(len(df_ttr))
ax.bar(x - 0.2, df_ttr["ttr"], 0.4, label="TTR", color="#0f766e")
ax.bar(x + 0.2, df_ttr["mattr_50"], 0.4, label="MATTR (ventana 50)", color="#2563eb", alpha=0.6)
ax.set_xticks(x)
ax.set_xticklabels(df_ttr["doc_id"], rotation=30, ha="right", fontsize=8)
ax.set_ylabel("Proporción de palabras únicas")
ax.set_title("El TTR cae con la longitud del documento; el MATTR se mantiene estable")
ax.legend(loc="upper left", bbox_to_anchor=(1.01, 1))
ax.spines[["top", "right"]].set_visible(False)
fig.text(0.99, 0.01, FUENTE, ha="right", fontsize=7)
plt.tight_layout()
plt.savefig(FIGS / "eda_ttr_mattr.png", dpi=200, bbox_inches="tight")
plt.show()

print(df_ttr.to_string(index=False))
print(f"Hallazgo: TTR de {df_ttr['ttr'].min():.2f} a {df_ttr['ttr'].max():.2f} (depende de la longitud); "
      f"MATTR de {df_ttr['mattr_50'].min():.2f} a {df_ttr['mattr_50'].max():.2f}.")
print("Decisión: comparar documentos con MATTR, no con TTR; lematizar solo para el baseline léxico y "
      "nunca antes de los embeddings.")

In [ ]:
# Caracteres especiales del texto extraído
chars_esp = Counter(ch for ch in corpus_all_text if ord(ch) > 127)
df_chars = pd.DataFrame([{"caracter": ch, "unicode": f"U+{ord(ch):04X}", "ocurrencias": n}
                         for ch, n in chars_esp.most_common()])
print(df_chars.head(20).to_string(index=False))
esperados = set("áéíóúÁÉÍÓÚñÑüÜ°º«»“”–—¿¡")
raros = df_chars[~df_chars["caracter"].isin(esperados)]
print(f"Hallazgo: {int(df_chars['ocurrencias'].sum())} caracteres no ASCII en {len(df_chars)} tipos; "
      f"{len(raros)} tipos fuera del español esperado.")
print("Decisión: la normalización NFC alcanza para los acentos; los tipos raros se revisan antes de indexar "
      "(no se reemplaza «Nº» por «No»).")

**Gráfico 7 — Similitud entre normas**

- **Variables:** similitud coseno TF-IDF entre cada par de normas (numérica continua, de 0 a 1).
- **Pregunta:** ¿hay normas casi iguales que el sistema podría confundir al buscar?
- **Gráfico:** mapa de calor con el valor en cada celda: es la forma de ver muchos pares a la vez. Más rojo, más parecidas.
- **Hallazgo y decisión:** se calculan con los datos y se imprimen debajo del gráfico.

In [ ]:
texts = []
labels = []
for payload in corpus:
    doc_id = payload["doc_id"]
    text = (CLEAN / f"{doc_id}.txt").read_text(encoding="utf-8")
    if text.strip():
        labels.append(doc_id)
        texts.append(text)

# Palabras vacías del español (lista corta, sin acentos para coincidir con strip_accents)
STOPWORDS_ES = """de la que el en y a los del se las por un para con una su al lo como mas o pero sus
le ya fue este ha si porque esta son entre cuando muy sobre ser tiene tambien me hasta hay donde quien
desde todo nos durante todos uno les contra otros ese eso ante ellos e esto mi antes algunos que unos
otro otras otra el tanto esa estos mucho quienes nada muchos cual poco ella estar estas algunas algo""".split()
# Las negaciones (no, sin, ni, nunca) NO se quitan: cambian el sentido de la norma.


def tfidf_sim(stop_words=None):
    v = TfidfVectorizer(lowercase=True, strip_accents="unicode", ngram_range=(1, 2),
                        min_df=1, stop_words=stop_words)
    return cosine_similarity(v.fit_transform(texts))


sim_raw = tfidf_sim()
sim = tfidf_sim(STOPWORDS_ES)
iu = np.triu_indices(len(labels), 1)
print(f"Similitud mediana entre pares: {np.median(sim_raw[iu]):.2f} con palabras vacías; "
      f"{np.median(sim[iu]):.2f} sin ellas.")

fig, ax = plt.subplots(figsize=(8, 7))
im = ax.imshow(sim, cmap="YlOrRd", vmin=0, vmax=1)
ax.set_xticks(range(len(labels)))
ax.set_xticklabels(labels, rotation=45, ha="right", fontsize=7)
ax.set_yticks(range(len(labels)))
ax.set_yticklabels(labels, fontsize=7)
for i in range(len(labels)):
    for j in range(len(labels)):
        ax.text(j, i, f"{sim[i, j]:.2f}", ha="center", va="center", fontsize=6,
                color="white" if sim[i, j] > 0.55 else "black")
ax.set_title(f"Sin palabras vacías, solo {int((np.triu(sim, 1) > 0.5).sum())} par(es) de normas supera 0,5 de similitud")
fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
plt.tight_layout()
fig.text(0.99, 0.01, FUENTE, ha="right", fontsize=7)
plt.savefig(FIGS / "eda_similitud.png", dpi=200, bbox_inches="tight")
plt.show()

off = sim.copy()
np.fill_diagonal(off, -1)
a, b = np.unravel_index(off.argmax(), off.shape)
pairs_05 = int((np.triu(sim, 1) > 0.5).sum())
def detectar_boilerplate(docs, umbral=0.4):
    """Líneas de más de 25 caracteres repetidas en al menos el 40 % de los documentos."""
    lineas = [l.strip() for d in docs for l in set(d.split("\n")) if len(l.strip()) > 25]
    return {l: c for l, c in Counter(lineas).items() if c / len(docs) >= umbral}


boilerplate = detectar_boilerplate(texts)
print(f"Boilerplate (≥ 40 % de los {len(texts)} documentos): {len(boilerplate)} líneas")
for l, c in sorted(boilerplate.items(), key=lambda x: -x[1])[:10]:
    print(f"  {c} docs · {l[:90]}")

print(f"Hallazgo: el par más similar es {labels[a]} / {labels[b]} (coseno {sim[a, b]:.2f}); "
      f"{pairs_05} pares superan 0.5.")
print(f"Hallazgo: con palabras vacías {int((np.triu(sim_raw, 1) > 0.5).sum())} pares superan 0.5: "
      "la similitud bruta está dominada por palabras funcionales.")
print("Decisión: quitar palabras vacías (no las negaciones) en el baseline léxico TF-IDF/BM25; no quitarlas "
      "antes de los embeddings. Los pares que siguen similares llevan preguntas de distinción en el conjunto de evaluación.")

### 5b. Normas incluidas en la página del Estatuto y texto duplicado

La página del Estatuto en el Digesto no trae solo el Estatuto: también incluye otras normas
relacionadas, y algunas de ellas también se descargan por separado. Sin este paso, ese texto
quedaría dos veces en el corpus.

Primero se corta la página en normas usando sus encabezados («Decreto Acuerdo Nº …», «Ley Nº … -
Decreto Nº …»). Después, para saber qué parte repite una norma descargada aparte, se usa el número
de la norma: si coincide, es la misma. La similitud de coseno (TF-IDF) y la proporción de palabras
compartidas lo confirman, pero solas no alcanzan: el Decreto Acuerdo 636/2021 y el Decreto 818/88
tratan los dos sobre interinatos y se parecen casi tanto como dos copias de una misma ley. Cuando
hay duplicado, se queda la versión descargada aparte.

Cada parte lleva también su estado. El Decreto 818/88 (derogado por el Decreto Acuerdo 2010/2013) y
el Decreto E. 541/02 (que la misma página marca como derogado) quedan como derogados. No se borran:
así el sistema puede reconocerlos y dejarlos de lado al responder.

**Gráficos 8 y 9 (en la celda siguiente)**

- **Gráfico 8 — Vigencia.** *Variables:* palabras por norma (numérica discreta) y estado (binaria:
  vigente o derogada). *Pregunta:* ¿cuánto del corpus ya no rige? *Gráfico:* barras horizontales
  ordenadas, en rojo las derogadas.
- **Gráfico 9 — Copias y normas parecidas.** *Variables:* similitud coseno (numérica continua) de
  cada par. *Pregunta:* ¿alcanza la similitud para saber si dos textos son la misma norma?
  *Gráfico:* barras horizontales: en verde los pares que son la misma norma, en rojo los pares más
  parecidos que son normas distintas, y una línea en 0,90.
- **Hallazgo y decisión:** al pie de la celda.

In [ ]:
HOST_ID = "estatuto_dl3122"
host_payload = next(p for p in corpus if p["doc_id"] == HOST_ID)
host_text = (CLEAN / f"{HOST_ID}.txt").read_text(encoding="utf-8")
host_man = df_manifest.set_index("doc_id").loc[HOST_ID]

# 1) Segmentación por encabezado de norma (admite la marca «-Derogado-»)
PATRON_NORMA = re.compile(
    r"(?m)^(?:Decreto(?:\s+Ley|\s+Acuerdo|\s+C\.\s*(?:y\s*)?E\.|\s+E\.)?|Ley)\s+N[°º]\s*[\d\.]+(?:/\d+)?"
    r"(?:\s*-\s*Decreto\s+N[°º]\s*\d+)?\s*\.?(?:\s*-Derogado-)?\s*$", re.I)
heads = list(PATRON_NORMA.finditer(host_text))
cortes = [0] + [m.start() for m in heads[1:]] + [len(host_text)]
segmentos = [host_text[a:b].strip() for a, b in zip(cortes[:-1], cortes[1:])]
encabezados = [m.group(0).strip() for m in heads]


def numero_norma(encabezado: str) -> str | None:
    m = re.search(r"N[°º]\s*([\d\.]+)", encabezado)
    return m.group(1).replace(".", "") if m else None


def numero_completo(encabezado: str) -> str:
    m = re.search(r"N[°º]\s*([\d\.]+(?:/\d+)?)", encabezado)
    return m.group(1) if m else ""


def tipo_norma(encabezado: str) -> str:
    e = encabezado.lower()
    if e.startswith("ley"):
        return "Ley"
    if "decreto ley" in e:
        return "Decreto Ley"
    if "decreto acuerdo" in e:
        return "Decreto Acuerdo"
    return "Decreto"


# Estado y relación con el Estatuto de cada segmento (decisiones registradas del responsable)
SEGMENTOS_DEROGADOS = {"818": "Derogado por el Decreto Acuerdo 2010/2013",
                       "541": "Marcado «-Derogado-» en el compilado (Decreto ECyT 756/2013)"}
RELACION_SEGMENTO = {"3122": "base", "3267": "modifica", "3564": "modifica", "5301": "modifica", "5160": "independiente"}  # el resto: reglamenta
print(f"Normas detectadas en el compilado ({HOST_ID}): {len(heads)}")
for k, h in enumerate(encabezados):
    print(f"  S{k:02d}  {h}{'  → Derogado' if numero_norma(h) in SEGMENTOS_DEROGADOS else ''}")

# 2) Pares por número de norma, con el segmento siguiente si el coseno sube 0,05 o más
individuales = {p["doc_id"]: (CLEAN / f"{p['doc_id']}.txt").read_text(encoding="utf-8")
                for p in corpus if p["doc_id"] != HOST_ID}
vec_dd = TfidfVectorizer(lowercase=True, strip_accents="unicode", ngram_range=(1, 2))
vec_dd.fit(list(individuales.values()) + segmentos)
coseno = lambda a, b: float(cosine_similarity(vec_dd.transform([a]), vec_dd.transform([b]))[0, 0])
sim_seg = cosine_similarity(vec_dd.transform(list(individuales.values())), vec_dd.transform(segmentos))


def palabras_distintas(t: str) -> set[str]:
    t = unicodedata.normalize("NFKD", t.lower())
    return set(re.findall(r"[a-zñ]{3,}|\d+", "".join(c for c in t if not unicodedata.combining(c))))


def cobertura(segmento: str, norma: str) -> float:
    """% de las palabras distintas del segmento presentes en la norma descargada aparte."""
    a = palabras_distintas(segmento)
    return 100 * len(a & palabras_distintas(norma)) / max(1, len(a))


UMBRAL_COSENO, UMBRAL_COBERTURA, MEJORA_ANEXO = 0.60, 80.0, 0.05
numero_de = df_normas.set_index("doc_id")["numero"].to_dict()
asociacion = {}
for d, texto in individuales.items():
    ks = [k for k, h in enumerate(encabezados) if numero_norma(h) == numero_de[d]]
    if not ks:
        continue
    k, j = ks[0], 1
    while k + j < len(segmentos) and coseno(texto, "\n".join(segmentos[k:k + j + 1])) >= coseno(texto, "\n".join(segmentos[k:k + j])) + MEJORA_ANEXO:
        j += 1
    seg = "\n".join(segmentos[k:k + j])
    asociacion[d] = {"segmentos": list(range(k, k + j)), "coseno": round(coseno(texto, seg), 3),
                     "cobertura": round(cobertura(seg, texto), 1),
                     "md5_igual": hashlib.md5(re.sub(r"\s+", " ", texto.lower()).encode()).hexdigest()
                                  == hashlib.md5(re.sub(r"\s+", " ", seg.lower()).encode()).hexdigest()}
df_asoc = pd.DataFrame(asociacion).T
print(df_asoc.to_string())
no_confirman = df_asoc[(df_asoc["coseno"] < UMBRAL_COSENO) | (df_asoc["cobertura"] < UMBRAL_COBERTURA)].index.tolist()
assert not no_confirman, f"Pares por número que no confirman (coseno ≥ {UMBRAL_COSENO}, cobertura ≥ {UMBRAL_COBERTURA} %): {no_confirman}"
duplicados = {s: d for d, v in asociacion.items() for s in v["segmentos"]}

# 3) Mapa del compilado
def estado_segmento(k: int) -> str:
    return "Derogado" if numero_norma(encabezados[k]) in SEGMENTOS_DEROGADOS else "Vigente"


mapa = pd.DataFrame([{
    "segmento": f"S{k:02d}", "norma": encabezados[k], "numero": numero_norma(encabezados[k]),
    "palabras": len(segmentos[k].split()), "estado": estado_segmento(k),
    "motivo_estado": SEGMENTOS_DEROGADOS.get(numero_norma(encabezados[k]), ""),
    "norma_descargada_aparte": duplicados.get(k, "—"),
    "se_indexa_desde_compilado": k not in duplicados,
} for k in range(len(segmentos))])
mapa.to_csv(ART / "mapa_compilado.csv", index=False)
print(mapa.to_string(index=False))

# 4) Corpus deduplicado, con los metadatos de vigencia y procedencia
man = df_manifest.set_index("doc_id")
corpus_dedup = []
for p in corpus:
    if p["doc_id"] == HOST_ID:
        continue
    m = man.loc[p["doc_id"]]
    corpus_dedup.append({"doc_id": p["doc_id"], "norma_id": m["norma_id"], "tipo": m["tipo"], "titulo": m["titulo"],
                         "estado": m["estado"], "estado_digesto": m["estado_digesto"], "fuente": "Digesto Catamarca",
                         "url_origen": m["url_origen"], "relacion_3122": m["relacion_3122"],
                         "articulo_3122": m["articulo_3122"], "origen": "norma", "formato": m["formato"],
                         "citable": True, "text": individuales[p["doc_id"]]})
for k in range(len(segmentos)):
    if k in duplicados:
        continue
    num = numero_norma(encabezados[k])
    corpus_dedup.append({"doc_id": f"{HOST_ID}__S{k:02d}", "norma_id": f"{tipo_norma(encabezados[k])} {numero_completo(encabezados[k])}",
                         "tipo": tipo_norma(encabezados[k]), "titulo": encabezados[k], "estado": estado_segmento(k),
                         "estado_digesto": f"{host_man['estado_digesto']} (página del compilado)",
                         "fuente": "Digesto Catamarca", "url_origen": host_man["url_origen"],
                         "relacion_3122": RELACION_SEGMENTO.get(num, "reglamenta"), "articulo_3122": "",
                         "origen": f"compilado, segmento S{k:02d}", "formato": "html",
                         "citable": True, "text": segmentos[k]})
with open(ART / "corpus_dedup.jsonl", "w", encoding="utf-8") as f:
    for rec in corpus_dedup:
        f.write(json.dumps(rec, ensure_ascii=False) + "\n")
df_dedup = pd.DataFrame(corpus_dedup).drop(columns="text")
df_dedup["palabras"] = [len(r["text"].split()) for r in corpus_dedup]

# 5) Controles
METADATOS = ["norma_id", "tipo", "estado", "estado_digesto", "fuente", "url_origen", "relacion_3122"]
metadatos_ok = bool(df_dedup[METADATOS].replace("", np.nan).notna().all().all())
derogados_ok = set(df_dedup.loc[df_dedup["estado"] == "Derogado", "norma_id"].str.extract(r"(\d+)")[0]) >= {"818", "541", "2010"}
print(f"\nCorpus deduplicado: {len(corpus_dedup)} normas o segmentos; {df_dedup['palabras'].sum()} palabras")
print("Pares confirmados:", {d: v["segmentos"] for d, v in asociacion.items()})
print("Metadatos de vigencia completos:", metadatos_ok, "| 818/88, 541/02 y 2010/2013 derogados:", derogados_ok)

# 6) Figura: palabras por norma del corpus deduplicado, según su estado
orden = df_dedup.sort_values("palabras")
fig, ax = plt.subplots(figsize=(10, 7))
ax.barh(orden["norma_id"] + "  (" + orden["origen"].str.replace("compilado, segmento ", "", regex=False) + ")",
        orden["palabras"], color=np.where(orden["estado"] == "Derogado", "#b91c1c", "#0f766e"))
ax.set_xscale("log")
ax.set_xlabel("Palabras (escala log)")
n_der = int((df_dedup["estado"] == "Derogado").sum())
pct_der = 100 * df_dedup.loc[df_dedup["estado"] == "Derogado", "palabras"].sum() / df_dedup["palabras"].sum()
ax.set_title(f"{n_der} de {len(df_dedup)} normas del corpus están derogadas ({pct_der:.0f} % de las palabras; en rojo)")
ax.spines[["top", "right"]].set_visible(False)
fig.text(0.99, 0.0, FUENTE, ha="right", fontsize=7)
plt.tight_layout()
plt.savefig(FIGS / "eda_vigencia.png", dpi=200, bbox_inches="tight")
plt.show()

# 7) Figura: los pares que son la misma norma frente a los pares más parecidos que son normas distintas
pares = [(f"{d} = S{v['segmentos'][0]:02d} (misma norma)", float(v["coseno"]), True) for d, v in asociacion.items()]
otros = sorted(((float(sim_seg[r, c]), d, c) for r, d in enumerate(individuales) for c in range(len(segmentos))
                if duplicados.get(c) != d), reverse=True)[:3]
pares += [(f"{d} ≠ S{c:02d} {encabezados[c]} (otra norma)", round(v, 3), False) for v, d, c in otros]
pares.sort(key=lambda p: p[1])
fig, ax = plt.subplots(figsize=(10, 4.5))
ax.barh([p[0] for p in pares], [p[1] for p in pares], color=["#0f766e" if p[2] else "#b91c1c" for p in pares])
for y, p in enumerate(pares):
    ax.text(p[1] + 0.01, y, f"{p[1]:.2f}", va="center", fontsize=8)
ax.axvline(0.90, color="#334155", linestyle="--", linewidth=1)
ax.text(0.905, -0.6, "0,90", fontsize=7, color="#334155")
ax.set_xlim(0, 1.08)
ax.set_xlabel("Similitud coseno TF-IDF (0 a 1)")
max_otro, min_par = max(p[1] for p in pares if not p[2]), min(p[1] for p in pares if p[2])
ax.set_title("Una norma distinta puede parecerse más que una copia: decide el número de norma" if max_otro > min_par
             else "Las copias se separan de las demás normas")
ax.spines[["top", "right"]].set_visible(False)
fig.text(0.99, 0.0, FUENTE, ha="right", fontsize=7)
plt.tight_layout()
plt.savefig(FIGS / "eda_dedup_coseno.png", dpi=200, bbox_inches="tight")
plt.show()

n_dup = len(asociacion)
print(f"Hallazgo: el compilado reúne {len(segmentos)} normas; {n_dup} también se descargan aparte; "
      f"{int(mapa['se_indexa_desde_compilado'].sum())} se indexan desde el compilado.")
print(f"Hallazgo: {n_der} normas del corpus están derogadas ({pct_der:.1f} % de las palabras).")
print("Decisión: indexar cada texto una sola vez (corpus_dedup.jsonl), con su estado, para que el sistema "
      "pueda excluir lo derogado al responder.")

**Gráfico 10 — Artículos por norma**

- **Variables:** menciones de «ARTICULO n» por norma (numérica discreta) y norma (categórica nominal).
- **Pregunta:** ¿el artículo es una marca que aparece en todas las normas, y sirve para cortar el texto?
- **Gráfico:** barras horizontales ordenadas, con el valor al lado de cada barra.
- **Hallazgo y decisión:** se calculan con los datos y se imprimen debajo del gráfico.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
df_art = df_eda.sort_values("articles", ascending=False)
ax.barh(df_art["doc_id"], df_art["articles"], color="#7c3aed")
ax.set_xlabel("Menciones de artículo detectadas")
for y, v in zip(df_art["doc_id"], df_art["articles"]):
    ax.text(v + 3, y, str(v), va="center", fontsize=7)
ax.set_title(f"{df_art.iloc[0]['doc_id']} reúne {df_art.iloc[0]['articles']} de las {df_art['articles'].sum()} menciones de artículo")
ax.spines[["top", "right"]].set_visible(False)
fig.text(0.99, 0.01, FUENTE, ha="right", fontsize=7)
ax.invert_yaxis()
plt.tight_layout()
plt.savefig(FIGS / "eda_articulos.png", dpi=200, bbox_inches="tight")
plt.show()

print(f"Hallazgo: el documento con más artículos es {df_art.iloc[0]['doc_id']} ({df_art.iloc[0]['articles']} menciones).")
print("Decisión: usar los artículos como separadores naturales del chunking.")

In [ ]:
findings = pd.DataFrame([
    {
        "hallazgo": f"{int((df_eda['tokens'] > 512).sum())} de {len(df_eda)} documentos superan 512 tokens de referencia.",
        "decision": "Aplicar chunking por estructura legal y medir con el tokenizador del modelo de embeddings.",
    },
    {
        "hallazgo": f"{int((~df_text['sin_texto']).sum())} de {len(df_text)} normas tienen texto extraíble ({int((df_text['formato'] == 'html').sum())} páginas HTML y {int((df_text['formato'] == 'pdf').sum())} PDF).",
        "decision": "Trabajar directamente con el texto extraído.",
    },
    {
        "hallazgo": f"{len(difieren)} norma(s) con un estado decidido distinto del de la ficha del Digesto.",
        "decision": "Guardar los dos campos (estado_digesto y estado) y usar el decidido para filtrar.",
    },
    {
        "hallazgo": f"{n_der} normas del corpus están derogadas ({pct_der:.1f} % de las palabras).",
        "decision": "Conservarlas con estado Derogado y excluirlas al responder.",
    },
    {
        "hallazgo": f"El documento con más artículos es {df_eda.sort_values('articles', ascending=False).iloc[0]['doc_id']}.",
        "decision": "Usar artículos, títulos y capítulos como separadores de chunk.",
    },
    {
        "hallazgo": f"{df_eda.iloc[0]['doc_id']} concentra el {df_eda.iloc[0]['pct_corpus']} % de los tokens.",
        "decision": "Estratificar las preguntas de evaluación y reportar métricas por documento para que el documento dominante no oculte fallas.",
    },
    {
        "hallazgo": f"Mediana de {df_art_tok['tokens'].median():.0f} tokens por artículo; "
                    f"{100 * (df_art_tok['tokens'] > 512).mean():.1f} % supera 512.",
        "decision": "Chunk por artículo y subdivisión solo de los artículos largos.",
    },
    {
        "hallazgo": f"El compilado ({HOST_ID}) reúne {len(segmentos)} normas; {n_dup} también se descargan aparte (identificadas por número).",
        "decision": "Limpieza de corpus: indexar los individuales y solo los segmentos no duplicados (corpus_dedup.jsonl).",
    },
    {
        "hallazgo": f"{int((df_art_tok['categoria'] == 'Requiere sub-chunking').sum())} artículos superan {UMBRAL_LARGO} tokens y "
                    f"{int((df_art_tok['categoria'] == 'Demasiado corto').sum())} tienen menos de {UMBRAL_CORTO}.",
        "decision": "Subdividir los artículos largos y agrupar los muy cortos con su contexto (título o capítulo).",
    },
    {
        "hallazgo": f"Las palabras vacías son el {df_sw['pct_vacias'].mean():.1f} % del texto.",
        "decision": "Quitarlas solo en el baseline léxico, conservando las negaciones; no antes de los embeddings.",
    },
    {
        "hallazgo": f"Frases de dominio frecuentes: {', '.join(NO_SEPARAR[:3])}.",
        "decision": "No cortar chunks dentro de estas frases.",
    },
    {
        "hallazgo": f"MATTR entre {df_ttr['mattr_50'].min():.2f} y {df_ttr['mattr_50'].max():.2f}; el TTR depende de la longitud.",
        "decision": "Comparar con MATTR; lematizar solo para el baseline léxico.",
    },
    {
        "hallazgo": f"{int(df_art_tok['modificado'].sum())} artículos tienen marca de modificación; DNI ocultos: {int(df_clean['dni_ocultos'].sum())}.",
        "decision": "Guardar la norma modificatoria y su fecha como metadato; no inferir vigencia automáticamente.",
    },
])
findings.to_csv(ART / "hallazgos_decisiones.csv", index=False)
findings

## 6. Entidades (NER) con expresiones regulares

Se buscan normas citadas, artículos, fechas, cargos, organismos, expedientes, modificaciones y
montos. Como el texto ya tiene los DNI ocultos, esos solo se cuentan. Se guarda una muestra para
revisarla a mano.

In [ ]:
PATTERNS = {
    "NORMA": r"\b(?:LEY|DECRETO(?:\s+LEY|\s+ACUERDO|\s+C\.\s*y\s*E\.)?)\s+N[°º]?\s*[\d\./]+",
    "ARTICULO": r"\bART(?:[IÍ]CULO|\.)\s*\d+[°º]?(?:\s*,?\s*inc(?:iso|\.)\s*\w+\)?)?",
    "FECHA": r"\b\d{1,2}\s+de\s+[A-Za-zÁÉÍÓÚáéíóúñ]+\s+(?:de\s+)?\d\.?\d{3}\b|\b\d{1,2}/\d{1,2}/\d{2,4}\b",
    "MODIFICACION": r"Modificad[oa]\s+por:?\s*(?:Art\.?\s*\d+\.?\s*)?Ley\s+N?[°º]?\s*[\d\.]+(?:\s*\((?:B\.?\s*O\.?\s*)?[\d/\.\s]+\))?",
    "CARGO": r"\b(?:Vicedirector|Director|Supervisor|Secretario|Regente|Preceptor|Maestro|Profesor)(?:es|as|a)?\b(?:\s+(?:titular|interino|suplente)(?:s|es)?)?",
    "ORGANISMO": r"(?:Ministerio de|Consejo General|Juntas? de Clasificaci[oó]n|Comit[eé] Ejecutivo)[^\n,;\.]{0,80}",
    "DNI": r"D\.?\s*N\.?\s*I\.?\s*N?[°º]?\.?\s*[\dX\.]{6,}",  # en el texto limpio, los dígitos ya son X
    "EXPEDIENTE": r"Expediente\s+N[°º]?\s*[\w\-/\.]+",
    "MONTO": r"\$\s?[\d\.\,]+|\b\d[\d\.]*\s*(?:pesos|unidades tributarias|UT)\b",
}

MESES = {
    "enero": "01", "febrero": "02", "marzo": "03", "abril": "04",
    "mayo": "05", "junio": "06", "julio": "07", "agosto": "08",
    "septiembre": "09", "setiembre": "09", "octubre": "10",
    "noviembre": "11", "diciembre": "12",
}


def normalize_date(value: str) -> str:
    m = re.match(r"(\d{1,2})\s+de\s+([A-Za-zÁÉÍÓÚáéíóúñ]+)\s+(?:de\s+)?(\d\.?\d{3})", value, re.I)
    if m:
        day, month, year = m.groups()
        year = year.replace(".", "")
        mm = MESES.get(month.lower(), "00")
        return f"{year}-{mm}-{int(day):02d}"
    m = re.match(r"(\d{1,2})/(\d{1,2})/(\d{2,4})", value)
    if m:
        day, month, year = m.groups()
        year = f"20{year}" if len(year) == 2 else year
        return f"{year}-{int(month):02d}-{int(day):02d}"
    return value


def normalize_entity(kind: str, value: str) -> str:
    v = re.sub(r"\s+", " ", value).strip()
    if kind == "FECHA":
        return normalize_date(v)
    if kind == "DNI":
        return re.sub(r"\d", "X", v)  # nunca se guarda un número de documento
    return v


def extract_entities(text: str, doc_id: str) -> list[dict]:
    rows = []
    for kind, pattern in PATTERNS.items():
        for m in re.finditer(pattern, text, re.I):
            rows.append({
                "doc_id": doc_id,
                "type": kind,
                "value": m.group(0),
                "normalized": normalize_entity(kind, m.group(0)),
                "start": m.start(),
                "end": m.end(),
            })
    return rows


entities = []
for payload in corpus:
    doc_id = payload["doc_id"]
    text = (CLEAN / f"{doc_id}.txt").read_text(encoding="utf-8")
    entities.extend(extract_entities(text, doc_id))

df_entities = pd.DataFrame(entities)
df_entities.to_csv(ART / "entities.csv", index=False)
df_entities.head(20)

**Gráfico 11 — Entidades encontradas**

- **Variables:** tipo de entidad (categórica nominal) y cantidad de menciones (numérica discreta).
- **Pregunta:** ¿qué entidades aparecen lo suficiente como para usarlas como filtros de búsqueda?
- **Gráfico:** barras horizontales ordenadas de mayor a menor, con el valor al lado.
- **Hallazgo y decisión:** se calculan con los datos y se imprimen debajo del gráfico.

In [ ]:
counts = df_entities.groupby("type").size().sort_values(ascending=False)
fig, ax = plt.subplots(figsize=(8, 4))
ax.barh(counts.index, counts.values, color="#be123c")
ax.set_xlabel("Menciones en el corpus")
for y, v in zip(counts.index, counts.values):
    ax.text(v * 1.01, y, str(v), va="center", fontsize=8)
ax.set_title(f"Domina {counts.index[0]} ({counts.iloc[0]} menciones); la menos frecuente es {counts.index[-1]} ({counts.iloc[-1]})")
ax.spines[["top", "right"]].set_visible(False)
fig.text(0.99, 0.01, FUENTE, ha="right", fontsize=7)
ax.invert_yaxis()
plt.tight_layout()
plt.savefig(FIGS / "eda_entidades.png", dpi=200, bbox_inches="tight")
plt.show()

print(f"Hallazgo: domina {counts.index[0]} ({counts.iloc[0]} menciones); "
      f"{df_entities.loc[df_entities['type'] == 'MODIFICACION', 'doc_id'].nunique()} documento(s) con modificaciones.")
print("Decisión: validar una muestra y filtrar falsos positivos antes de usar las entidades como metadatos.")

In [ ]:
sample = df_entities.groupby("type", group_keys=False).head(5).copy()
sample["correcto"] = ""
sample.to_csv(ART / "entities_validation_sample.csv", index=False)
print("Muestra para validación manual guardada en artifacts/entities_validation_sample.csv")
sample

## 7. Exportación y controles

Se guardan los archivos que usa la etapa siguiente y, al final, una tabla con los controles
automáticos. Todos tienen que dar `True`.

In [ ]:
corpus_jsonl = []
for payload in corpus:
    doc_id = payload["doc_id"]
    m = df_manifest.set_index("doc_id").loc[doc_id]
    corpus_jsonl.append({
        "doc_id": doc_id, "norma_id": m["norma_id"], "titulo": m["titulo"], "url_origen": m["url_origen"],
        "estado": m["estado"], "estado_digesto": m["estado_digesto"], "formato": m["formato"],
        "pages": len(payload["pages"]), "text": (CLEAN / f"{doc_id}.txt").read_text(encoding="utf-8"),
    })

with open(ART / "corpus.jsonl", "w", encoding="utf-8") as f:
    for rec in corpus_jsonl:
        f.write(json.dumps(rec, ensure_ascii=False) + "\n")

print("Artefactos generados:")
for p in sorted(ART.glob("*")):
    print("-", p.name)
for p in sorted(FIGS.glob("*")):
    print("-", p.name)

In [ ]:
# Ningún DNI con dígitos en los artefactos del corpus
DNI_DIGITOS = re.compile(r"\b(?:D\.?\s?N\.?\s?I\.?|M\.?\s?I\.?|L\.?\s?C\.?)\s*(?:N[°º]\.?|Nro\.?)?\s*:?\s*\d{1,2}\.?\d{3}\.?\d{3}\b", re.I)
artefactos_texto = list(CLEAN.glob("*.txt")) + [ART / "corpus_dedup.jsonl", ART / "corpus.jsonl",
                                               ART / "entities.csv", ART / "entities_validation_sample.csv"]
con_dni = [p.name for p in artefactos_texto if DNI_DIGITOS.search(p.read_text(encoding="utf-8"))]
urls = df_manifest["url_origen"].tolist()

checks = {
    "13 normas de la instantánea con su SHA-256": len(df_manifest) == EXPECTED_DOCS and all(hash_inst.values()),
    "Título esperado en cada norma": bool(df_manifest["titulo_esperado"].all()),
    "Estado de la ficha leído en las 13 normas": bool(df_manifest["estado_digesto"].notna().all()),
    "Texto extraído en las 13 normas": not df_text["sin_texto"].any(),
    "Sin restos de la interfaz del sitio": int(df_text["restos_interfaz"].sum()) == 0,
    "Fuente: solo el Digesto": all(u.startswith(DIGESTO) for u in urls),
    "3 o más gráficos EDA": len(list(FIGS.glob("eda_*.png"))) >= 3,
    "Entidades extraídas": len(df_entities) > 0,
    "Pérdida de limpieza ≤ 5 %": bool((df_clean["pct_lost"] <= MAX_LOSS_PCT).all()),
    "Limpieza preserva negaciones, artículos y fechas": bool(df_clean["preserva_todo"].all()),
    "Ningún DNI con dígitos en los artefactos": not con_dni,
    "Compilado segmentado en normas": len(segmentos) >= 10,
    "Duplicados por número confirmados (coseno y cobertura)": not no_confirman and len(asociacion) >= 4,
    "818/88, 541/02 y 2010/2013 marcados como derogados": derogados_ok,
    "Metadatos de vigencia completos": metadatos_ok,
    "Mapa del compilado y corpus deduplicado": (ART / "mapa_compilado.csv").exists() and (ART / "corpus_dedup.jsonl").exists(),
    "Manifiesto con hashes": (ART / "manifest.csv").exists() and (ART / "manifest.json").exists(),
}
print("Artefactos con DNI:", con_dni or "ninguno")
pd.Series(checks)

### Para revisar después de ejecutar

- [ ] Los 13 hashes coinciden con el manifiesto, cada norma tiene su título y su estado.
- [ ] El control contra el Digesto se informó (igual, cambió o sin acceso).
- [ ] Todas las normas tienen texto y no quedan restos del sitio (menú, comentarios).
- [ ] La limpieza pierde menos del 5 % y no toca negaciones, artículos ni fechas.
- [ ] Ningún archivo generado tiene un DNI con números.
- [ ] Cada gráfico termina con una decisión calculada a partir de los datos.
- [ ] La página del Estatuto quedó separada en normas, los duplicados se confirmaron y el 818/88,
      el 541/02 y el 2010/2013 figuran como derogados.
- [ ] Están la tabla de entidades y la muestra para revisar a mano.

## 8. Lo que queda para la próxima etapa

Este notebook no parte las normas en fragmentos (*chunking*), no calcula embeddings, no arma el
índice de búsqueda ni genera respuestas. Todo eso empieza desde `corpus_dedup.jsonl`.